# API T-BANL

Исследование  API Т-Банка для выгрузки дивидендов

## Дивиденды

Цель:
- проверить доступность истории дивидендов;
- изучить структуру данных;
- получить данные по выбранным акциям;
- определить поля, необходимые для хранения.


In [51]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

from moex_analytics.settings import PROJECT_ROOT

load_dotenv(PROJECT_ROOT / ".env")
T_TOKEN = os.environ["T_TOKEN"]

PROJECT_ROOT = Path.cwd().parents[1]
CERT_PATH = str(PROJECT_ROOT / "certs/russian-trusted-ca.pem")

In [52]:
# Полуить список всех инструментов.

def get_list_all_inst():
  url = (
      "https://sandbox-invest-public-api.tbank.ru:443/rest/"
      "tinkoff.public.invest.api.contract.v1.InstrumentsService/Shares"
  )
  payload = {
    "instrumentStatus": "INSTRUMENT_STATUS_BASE",
    "instrumentExchange": "INSTRUMENT_EXCHANGE_UNSPECIFIED"
  }

  headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f'Bearer {T_TOKEN}'
  }

  response = requests.post(
      url=url,
      json=payload,
      headers=headers,
      timeout=30,
      verify=CERT_PATH
  )
  return response.json()


In [83]:
# Получить информацию об инструменте.

def find_instrument(ticker):
  url = (
      "https://sandbox-invest-public-api.tbank.ru:443/rest/"
      "tinkoff.public.invest.api.contract.v1.InstrumentsService/FindInstrument"
  )
  payload = {
    "query": f"{ticker}",
    "instrumentKind": "INSTRUMENT_TYPE_SHARE",
    "apiTradeAvailableFlag": True
  }

  headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f'Bearer {T_TOKEN}'
  }

  response = requests.post(
      url=url,
      json=payload,
      headers=headers,
      timeout=30,
      verify=CERT_PATH
  )
  return response.json()

In [ ]:
 # Выгрузить дивиденды за период
def dividends(instrumentId):
  url = (
      "https://sandbox-invest-public-api.tbank.ru:443/rest/"
      "tinkoff.public.invest.api.contract.v1.InstrumentsService/GetDividends"
  )
  payload = {
    "from": "2026-01-24T16:08:58.701Z",
    "to": "2026-09-24T16:08:58.701Z",
    "instrumentId": f"{instrumentId}"
  }

  headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': f'Bearer {T_TOKEN}'
  }

  response = requests.post(
      url=url,
      json=payload,
      headers=headers,
      timeout=30,
      verify=CERT_PATH
  )
  return response.json()

In [88]:
ticker = "YDEX"
instrument = find_instrument(ticker)['instruments'][0]
uid = instrument['uid']
instrument

{'isin': 'RU000A107T19',
 'figi': 'TCS00A107T19',
 'ticker': 'YDEX',
 'classCode': 'TQBR',
 'instrumentType': 'share',
 'name': 'Яндекс',
 'uid': '7de75794-a27f-4d81-a39b-492345813822',
 'positionUid': '205a3042-12d0-4b21-ae8a-7764ff2260cc',
 'instrumentKind': 'INSTRUMENT_TYPE_SHARE',
 'apiTradeAvailableFlag': True,
 'forIisFlag': True,
 'first1minCandleDate': '2018-03-07T18:38:00Z',
 'first1dayCandleDate': '2014-06-04T07:00:00Z',
 'forQualInvestorFlag': False,
 'weekendFlag': True,
 'blockedTcaFlag': False,
 'lot': 1}

In [91]:
div = dividends(uid)
div["dividends"]

[{'dividendNet': {'currency': 'rub', 'units': '110', 'nano': 0},
  'paymentDate': '2026-09-24T00:00:00Z',
  'declaredDate': '2026-09-19T00:00:00Z',
  'lastBuyDate': '2026-09-18T00:00:00Z',
  'dividendType': '',
  'recordDate': '2026-09-21T00:00:00Z',
  'regularity': 'Semi-Anl',
  'closePrice': {'currency': 'rub', 'units': '3790', 'nano': 500000000},
  'yieldValue': {'units': '2', 'nano': 970000000},
  'createdAt': '2026-09-24T06:48:36.032347Z'},
 {'dividendNet': {'currency': 'rub', 'units': '110', 'nano': 0},
  'paymentDate': '2026-04-29T00:00:00Z',
  'declaredDate': '2026-04-25T00:00:00Z',
  'lastBuyDate': '2026-04-24T00:00:00Z',
  'dividendType': '',
  'recordDate': '2026-04-27T00:00:00Z',
  'regularity': 'Annual',
  'closePrice': {'currency': 'rub', 'units': '4251', 'nano': 500000000},
  'yieldValue': {'units': '2', 'nano': 530000000},
  'createdAt': '2026-04-29T05:50:19.557199Z'}]

In [ ]:
selected_securities = [
    "SBER",
    "GAZP",
    "VTBR",
    "T",
    "LKOH",
    "NVTK",
    "ROSN",
    "GMKN",
    "YDEX",
    "PLZL",
    "SMLT",
    "OZON",
    "X5",
    "RUAL",
    "TATN",
    "AFKS",
    "MOEX",
    "PIKK",
    "POSI",
    "AQUA",
]

dividents_list = {}

for ticker in selected_securities:
    instrument = find_instrument(ticker)['instruments'][0]
    uid = instrument['uid']
    div = dividends(uid)
    if div['dividends']:
         dividents_list[ticker] = div



In [93]:
dividents_list

{'SBER': {'dividends': [{'dividendNet': {'currency': 'rub',
     'units': '37',
     'nano': 640000000},
    'paymentDate': '2026-08-04T00:00:00Z',
    'declaredDate': '2026-07-18T00:00:00Z',
    'lastBuyDate': '2026-07-17T00:00:00Z',
    'dividendType': '',
    'recordDate': '2026-07-20T00:00:00Z',
    'regularity': 'Annual',
    'closePrice': {'currency': 'rub', 'units': '277', 'nano': 230000000},
    'yieldValue': {'units': '13', 'nano': 490000000},
    'createdAt': '2026-08-03T23:41:32.515407Z'}]},
 'GAZP': {'dividends': []},
 'VTBR': {'dividends': [{'dividendNet': {'currency': 'rub',
     'units': '9',
     'nano': 710000000},
    'paymentDate': '2026-08-04T00:00:00Z',
    'declaredDate': '2026-07-18T00:00:00Z',
    'lastBuyDate': '2026-07-17T00:00:00Z',
    'dividendType': '',
    'recordDate': '2026-07-20T00:00:00Z',
    'regularity': 'Annual',
    'closePrice': {'currency': 'rub', 'units': '58', 'nano': 550000000},
    'yieldValue': {'units': '16', 'nano': 820000000},
    'crea